# ❤️‍🩹 HEARTBREAK AI V2 — INTERACTIVE INFERENCE & TESTING NOTEBOOK

Notebook ini digunakan untuk menguji model **Heartbreak Severity Classifier V2** secara interaktif.

### ✨ 3 Tingkat Keparahan (3-Tier Clinical Severity Scale):
1. 🟢 **RINGAN (Keparahan Rendah / Adaptif)**: Probabilitas Distres < 35%
2. 🟡 **SEDANG (Keparahan Moderat / Fase Transisi)**: Probabilitas Distres 35% – 75%
3. 🔴 **BERAT (Keparahan Tinggi / Distres Akut)**: Probabilitas Distres ≥ 75%


## 📦 STEP 0 — Setup Environment & Load Model Bundle


In [ ]:
# CELL 0: SETUP ENVIRONMENT & LOAD MODEL BUNDLE V2
import os
import joblib
import numpy as np
import pandas as pd
import re

print("📥 Memuat Model Bundle V2...\n")

BUNDLE_PATH = 'heartbreak_demographic_bundle_v2.pkl'
DRIVE_PATH = '/content/drive/MyDrive/heartbreak_demographic_bundle_v2.pkl'

if os.path.exists(BUNDLE_PATH):
    bundle = joblib.load(BUNDLE_PATH)
    print(f"✅ Berhasil memuat bundle dari direktori lokal: {BUNDLE_PATH}")
elif os.path.exists(DRIVE_PATH):
    bundle = joblib.load(DRIVE_PATH)
    print(f"✅ Berhasil memuat bundle dari Google Drive: {DRIVE_PATH}")
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        bundle = joblib.load(DRIVE_PATH)
        print(f"✅ Berhasil memuat bundle setelah mount Drive: {DRIVE_PATH}")
    except Exception as e:
        raise FileNotFoundError("❌ File heartbreak_demographic_bundle_v2.pkl tidak ditemukan. Pastikan file berada di direktori kerja atau Google Drive.")

model = bundle['model']
scaler = bundle['scaler']
feature_names = bundle['feature_names']
label_decoder = bundle['label_decoder']
default_values = bundle['default_values']
meta = bundle['metadata']

print("\n📋 Metadata Bundle:")
print(f"   • Versi Model      : {meta.get('version', '2.0.0')}")
print(f"   • Arsitektur Model : {meta.get('model_architecture', 'Ensemble')}")
print(f"   • Test Accuracy    : {meta.get('metrics', {}).get('test_accuracy', 0.0)}%")
print(f"   • Test ROC-AUC     : {meta.get('metrics', {}).get('test_roc_auc', 0.0)}")


## ⚙️ STEP 1 — Modul Auto-Converter Durasi Alami (Hari/Minggu/Bulan/Tahun)


In [ ]:
# CELL 1: AUTO-CONVERTER DURASI NATURAL KE SATUAN BULAN & KATEGORI ORDINAL

def convert_ke_bulan(nilai: float, satuan: str) -> float:
    """
    Mengonversi nilai durasi dari satuan apa pun ke satuan bulan.
    Satuan yang didukung: hari, minggu, bulan, tahun (case-insensitive).
    """
    satuan = str(satuan).strip().lower()
    konversi = {
        'hari': 1.0 / 30.0,
        'hari-hari': 1.0 / 30.0,
        'day': 1.0 / 30.0,
        'days': 1.0 / 30.0,
        'minggu': 1.0 / 4.0,
        'week': 1.0 / 4.0,
        'weeks': 1.0 / 4.0,
        'bulan': 1.0,
        'month': 1.0,
        'months': 1.0,
        'tahun': 12.0,
        'year': 12.0,
        'years': 12.0
    }
    if satuan not in konversi:
        raise ValueError(f"Satuan '{satuan}' tidak valid! Gunakan: hari, minggu, bulan, atau tahun.")
    return float(nilai) * konversi[satuan]

def kategori_lama_hubungan(durasi_bulan: float) -> str:
    """Memetakan durasi hubungan dalam bulan ke kategori ordinal dataset."""
    if durasi_bulan < 6.0:
        return '< 6 bulan'
    elif durasi_bulan < 12.0:
        return '6 bulan - 1 tahun'
    elif durasi_bulan < 36.0:
        return '1 - 3 tahun'
    elif durasi_bulan < 60.0:
        return '3 - 5 tahun'
    else:
        return '> 5 tahun'

def kategori_sejak_putus(durasi_bulan: float) -> str:
    """Memetakan durasi sejak putus dalam bulan ke kategori ordinal dataset."""
    if durasi_bulan < 1.0:
        return '< 1 bulan'
    elif durasi_bulan < 3.0:
        return '1 - 3 bulan'
    elif durasi_bulan < 6.0:
        return '3 - 6 bulan'
    elif durasi_bulan < 12.0:
        return '6 - 12 bulan'
    else:
        return '> 1 tahun'

print("✅ Modul konversi durasi alami siap digunakan!")


## 🧠 STEP 2 — Pipeline Preprocessing & Feature Transformer


In [ ]:
# CELL 2: PREPROCESSING INPUT PENGGUNA KE VEKTOR FITUR TERSTANDAR

def clean_column_name(col_name):
    """Standardisasi nama kolom encoder."""
    col_name = str(col_name).strip()
    col_name = re.sub(r'[<>]+', '', col_name)
    col_name = re.sub(r'[?.,!()]+', '', col_name)
    col_name = re.sub(r'\s+-\s+', '_', col_name)
    col_name = re.sub(r'\s+', '_', col_name)
    col_name = re.sub(r'_+', '_', col_name)
    return col_name.strip('_')

def preprocess_user_input(
    umur: float,
    lama_hubungan_nilai: float,
    lama_hubungan_satuan: str,
    sejak_putus_nilai: float,
    sejak_putus_satuan: str,
    jenis_kelamin: str = None,
    pendidikan: str = None,
    siapa_mengakhiri: str = None,
    masih_komunikasi: str = None,
    frekuensi_medsos: str = None
) -> tuple:
    """
    Mengonversi input pengguna (wajib + opsional) menjadi DataFrame 1 baris
    yang telah melewati feature engineering, one-hot encoding, dan scaling.
    """
    durasi_hubungan_bulan = convert_ke_bulan(lama_hubungan_nilai, lama_hubungan_satuan)
    durasi_putus_bulan = convert_ke_bulan(sejak_putus_nilai, sejak_putus_satuan)
    
    kat_lama_hubungan = kategori_lama_hubungan(durasi_hubungan_bulan)
    kat_sejak_putus = kategori_sejak_putus(durasi_putus_bulan)
    
    jk = jenis_kelamin if jenis_kelamin is not None else default_values.get('Jenis Kelamin', 'Perempuan')
    pend = pendidikan if pendidikan is not None else default_values.get('Pendidikan', 'S1')
    pengakhiri = siapa_mengakhiri if siapa_mengakhiri is not None else default_values.get('Siapa yang Mengakhiri Hubungan?', 'Pasangan yang mengakhiri')
    komunikasi = masih_komunikasi if masih_komunikasi is not None else default_values.get('Apakah Masih Berkomunikasi dengan Mantan?', 'Tidak sama sekali')
    medsos = frekuensi_medsos if frekuensi_medsos is not None else default_values.get('Seberapa Sering Melihat Media Sosial Mantan?', 'Jarang')
    
    recovery_ratio = durasi_putus_bulan / (durasi_hubungan_bulan + 1e-5)
    log_recovery_index = np.log1p(durasi_putus_bulan) / np.log1p(durasi_hubungan_bulan)
    umur_mulai_hubungan = max(10.0, float(umur) - (durasi_hubungan_bulan / 12.0))
    
    feature_dict = {feat: 0.0 for feat in feature_names}
    
    feature_dict['Umur'] = float(umur)
    feature_dict['durasi_hubungan_bulan'] = float(durasi_hubungan_bulan)
    feature_dict['durasi_putus_bulan'] = float(durasi_putus_bulan)
    feature_dict['recovery_ratio'] = float(recovery_ratio)
    feature_dict['log_recovery_index'] = float(log_recovery_index)
    feature_dict['umur_mulai_hubungan'] = float(umur_mulai_hubungan)
    
    active_categorical_pairs = [
        ('Jenis Kelamin', jk),
        ('Pendidikan', pend),
        ('Lama Hubungan Sebelum Putus', kat_lama_hubungan),
        ('Sudah Berapa Lama Sejak Putus?', kat_sejak_putus),
        ('Siapa yang Mengakhiri Hubungan?', pengakhiri),
        ('Apakah Masih Berkomunikasi dengan Mantan?', komunikasi),
        ('Seberapa Sering Melihat Media Sosial Mantan?', medsos)
    ]
    
    for col, val in active_categorical_pairs:
        clean_feat_name = clean_column_name(f"{col}_{val}")
        if clean_feat_name in feature_dict:
            feature_dict[clean_feat_name] = 1.0
        else:
            for fn in feature_names:
                if clean_column_name(str(col)) in fn and clean_column_name(str(val)) in fn:
                    feature_dict[fn] = 1.0
                    break
    
    df_single = pd.DataFrame([feature_dict])[feature_names]
    df_single_scaled = pd.DataFrame(scaler.transform(df_single), columns=feature_names)
    
    return df_single_scaled, {
        'durasi_hubungan_bulan': durasi_hubungan_bulan,
        'durasi_putus_bulan': durasi_putus_bulan,
        'kat_lama_hubungan': kat_lama_hubungan,
        'kat_sejak_putus': kat_sejak_putus,
        'recovery_ratio': recovery_ratio,
        'is_fallback_used': any(x is None for x in [jenis_kelamin, pendidikan, siapa_mengakhiri, masih_komunikasi, frekuensi_medsos])
    }

print("✅ Pipeline preprocessor & transformer siap digunakan!")


## 🩺 STEP 3 — Fungsi Inferensi Utama (Dengan 3-Tier Severity: Ringan, Sedang, Berat)


In [ ]:
# CELL 3: FUNGSI INFERENSI DENGAN KLASIFIKASI 3-TIER (RINGAN, SEDANG, BERAT)

def predict_heartbreak_severity(
    nama: str,
    umur: float,
    lama_hubungan_nilai: float,
    lama_hubungan_satuan: str,
    sejak_putus_nilai: float,
    sejak_putus_satuan: str,
    jenis_kelamin: str = None,
    pendidikan: str = None,
    siapa_mengakhiri: str = None,
    masih_komunikasi: str = None,
    frekuensi_medsos: str = None,
    tampilkan_detail: bool = True
) -> dict:
    """
    Memprediksi tingkat keparahan patah hati (Ringan / Sedang / Berat)
    berdasarkan probabilitas terkalibrasi model AI dan memberikan rekomendasi klinis personal.
    """
    # 1. Preprocessing Input
    X_input_scaled, info = preprocess_user_input(
        umur=umur,
        lama_hubungan_nilai=lama_hubungan_nilai,
        lama_hubungan_satuan=lama_hubungan_satuan,
        sejak_putus_nilai=sejak_putus_nilai,
        sejak_putus_satuan=sejak_putus_satuan,
        jenis_kelamin=jenis_kelamin,
        pendidikan=pendidikan,
        siapa_mengakhiri=siapa_mengakhiri,
        masih_komunikasi=masih_komunikasi,
        frekuensi_medsos=frekuensi_medsos
    )
    
    # 2. Prediksi Probabilitas Terkalibrasi
    pred_proba = model.predict_proba(X_input_scaled)[0]
    prob_ringan = pred_proba[0] * 100
    prob_distres = pred_proba[1] * 100 if len(pred_proba) > 1 else (100 - prob_ringan)
    
    # 3. Klasifikasi 3-Tier Severity (Clinical Thresholding)
    if prob_distres >= 75.0:
        pred_label = 'Berat'
        pred_class = 2
        badge_color = '🔴'
        status_desc = 'Keparahan Patah Hati Tinggi / Akut (Distres Emosional Intensif)'
        saran = [
            'Prioritas Utama: Sangat dianjurkan berkonsultasi dengan psikolog atau konselor profesional untuk pendampingan reguler.',
            'Terapkan STRICT NO-CONTACT: Blokir/mute semua akses media sosial mantan untuk memutus siklus distres.',
            'Jangan menahan beban sendirian; libatkan keluarga atau support system terdekat yang aman dan suportif.',
            'Jaga kebutuhan fisik esensial: istirahat cukup, hindari isolasi diri berkepanjangan, dan tunda keputusan hidup yang besar.'
        ]
    elif prob_distres >= 35.0:
        pred_label = 'Sedang'
        pred_class = 1
        badge_color = '🟡'
        status_desc = 'Keparahan Patah Hati Moderat (Fase Transisi & Adaptasi Emosional)'
        saran = [
            'Terapkan aturan No-Contact (batasi komunikasi dan hindari stalking media sosial mantan).',
            'Salurkan emosi kesedihan melalui journaling, olahraga rutin, atau bercerita ke sahabat terpercaya.',
            'Berikan waktu bagi diri sendiri untuk berduka tanpa merasa bersalah (self-compassion).'
        ]
    else:
        pred_label = 'Ringan'
        pred_class = 0
        badge_color = '🟢'
        status_desc = 'Keparahan Patah Hati Rendah (Adaptif, Stabil, & Pulih)'
        saran = [
            'Pertahankan rutinitas positif harian dan aktivitas produktif yang sedang berjalan.',
            'Fokus pada pengembangan diri, hobi baru, dan pencapaian target masa depan.',
            'Buka diri secara perlahan untuk memperluas lingkaran sosial yang sehat.'
        ]
    
    result = {
        'nama': nama,
        'prediksi_kelas': pred_class,
        'kategori_severity': pred_label,
        'probabilitas_ringan': prob_ringan,
        'probabilitas_distres': prob_distres,
        'info_durasi': info,
        'saran_rekomendasi': saran
    }
    
    if tampilkan_detail:
        print('=' * 65)
        print('❤️‍🩹 LAPORAN ANALISIS KEPARAHAN PATAH HATI — HEARTBREAK AI V2')
        print('=' * 65)
        print(f"👤 Responden            : {nama} ({int(umur)} tahun)")
        print(f"⏳ Durasi Hubungan       : {lama_hubungan_nilai} {lama_hubungan_satuan} (~{info['durasi_hubungan_bulan']:.1f} bulan | '{info['kat_lama_hubungan']}')")
        print(f"💔 Durasi Sejak Putus    : {sejak_putus_nilai} {sejak_putus_satuan} (~{info['durasi_putus_bulan']:.1f} bulan | '{info['kat_sejak_putus']}')")
        print(f"🔄 Rasio Pemulihan       : {info['recovery_ratio']:.4f}")
        if info['is_fallback_used']:
            print('ℹ️ Catatan Input         : Menggunakan fallback default untuk field opsional yang dikosongkan.')
        print('-' * 65)
        print(f"{badge_color} TINGKAT KEPARAHAN     : {pred_label.upper()} ({status_desc})")
        print(f"📊 Indeks Distres / Skor : Distres = {prob_distres:.1f}% | Kestabilan = {prob_ringan:.1f}%")
        print('-' * 65)
        print('💡 Rekomendasi Pemulihan:')
        for i, tip in enumerate(saran, 1):
            print(f"   {i}. {tip}")
        print('=' * 65 + '\n')
    
    return result

print('✅ Fungsi inferensi 3-Tier predict_heartbreak_severity berhasil diupdate!')


## 🧪 STEP 4 — Skenario Pengujian Interaktif (Termasuk Kategori BERAT, SEDANG, RINGAN)


In [ ]:
# CELL 4.1: SKENARIO KEPARAHAN TINGGI (KATEGORI BERAT / DISTRES AKUT)
print("🧪 SKENARIO KEPARAHAN BERAT: Pacaran 6 Tahun, Baru Putus 2 Hari, Diputuskan Pasangan, Masih Stalking Medsos Sering\n")

res_berat = predict_heartbreak_severity(
    nama='Dimas Anggara',
    umur=22,
    lama_hubungan_nilai=6,
    lama_hubungan_satuan='tahun',     # 6 tahun pacaran (sangat lama)
    sejak_putus_nilai=2,
    sejak_putus_satuan='hari',        # baru 2 hari putus (sangat baru)
    jenis_kelamin='Laki-laki',
    pendidikan='S1',
    siapa_mengakhiri='Pasangan yang mengakhiri',
    masih_komunikasi='Sering',
    frekuensi_medsos='Sering'
)


In [ ]:
# CELL 4.2: SKENARIO KEPARAHAN MODERAT (KATEGORI SEDANG)
print("🧪 SKENARIO KEPARAHAN SEDANG: Pacaran 2 Tahun, Putus 2 Bulan Lalu, Masih Transisi Emosional\n")

res_sedang = predict_heartbreak_severity(
    nama='Budi Pratama',
    umur=23,
    lama_hubungan_nilai=2,
    lama_hubungan_satuan='tahun',
    sejak_putus_nilai=2,
    sejak_putus_satuan='bulan',
    jenis_kelamin='Laki-laki',
    pendidikan='S1',
    siapa_mengakhiri='Pasangan yang mengakhiri',
    masih_komunikasi='Kadang-kadang',
    frekuensi_medsos='Kadang-kadang'
)


In [ ]:
# CELL 4.3: SKENARIO KEPARAHAN RENDAH (KATEGORI RINGAN / RECOVERED)
print("🧪 SKENARIO KEPARAHAN RINGAN: Pacaran 2 Tahun, Sudah 2 Tahun Sejak Putus (Stabil & Move On)\n")

res_ringan = predict_heartbreak_severity(
    nama='Rian Ardiansyah',
    umur=25,
    lama_hubungan_nilai=2,
    lama_hubungan_satuan='tahun',
    sejak_putus_nilai=2,
    sejak_putus_satuan='tahun',
    jenis_kelamin='Laki-laki',
    pendidikan='S1',
    siapa_mengakhiri='Keputusan bersama',
    masih_komunikasi='Tidak sama sekali',
    frekuensi_medsos='Tidak pernah'
)


In [ ]:
# CELL 4.4: SKENARIO INPUT MINIMAL (HANYA WAJIB, OPSIONAL = NONE)
print("🧪 SKENARIO INPUT MINIMAL: Hanya Mengisi Umur, Lama Hubungan, & Sejak Putus\n")

res_minimal = predict_heartbreak_severity(
    nama='Siti Rahmawati',
    umur=21,
    lama_hubungan_nilai=1.5,
    lama_hubungan_satuan='tahun',
    sejak_putus_nilai=5,
    sejak_putus_satuan='bulan',
    jenis_kelamin=None,
    pendidikan=None,
    siapa_mengakhiri=None,
    masih_komunikasi=None,
    frekuensi_medsos=None
)

print('\n🎉 SELURUH PENGUJIAN TINGKAT KEPARAHAN (RINGAN, SEDANG, BERAT) SELESAI & BERFUNGSI SEMPURNA!')
